# 데이터 불러오기

In [2]:
# 라이브러리
import pandas as pd
import numpy as np
import re

In [3]:
# 추천순 리뷰 raw 데이터 로드
df = pd.read_csv("musinsastandard_recommend_reviews_raw.csv")
df_products = pd.read_csv("musinsastandard_woman_recommend_top50.csv")

print(f"리뷰 수: {len(df)}건")
print(f"컬럼: {df.columns.tolist()}")
print(df.head(3))

리뷰 수: 21295건
컬럼: ['goods_no', 'review_no', 'content', 'grade', 'option_size', 'size_feedback', 'create_date', 'review_sex', 'height', 'weight']
   goods_no  review_no                                            content  \
0   6092187   83840402  커브드팬츠 베이지 색상까지 구매했습니다. 스트레이트 핏이 돌아온다고 하지만 아직 널...   
1   6092187   83830897  매우 잘 구겨져서... 주의...ㅠㅠ\n또 여름에 입기에는 더울 것 같아요.\n지금...   
2   6092187   83829268  너무 마음에 듭니다. 모양이 화면 그대로 이쁘고  무엇보다 기장이 짧지 않아서 좋고...   

   grade  option_size size_feedback                    create_date review_sex  \
0      5  03.베이지 · 26          정사이즈  2026-04-21T02:59:07.000+09:00         여성   
1      4   01.블랙 · 26          조금 큼  2026-04-20T18:50:23.000+09:00         여성   
2      5  03.베이지 · 29          정사이즈  2026-04-20T17:33:29.000+09:00         여성   

   height  weight  
0     158      59  
1     158      54  
2     169      59  


In [8]:
# 겹치는 15개 goods_no
overlap_goods = [6092190, 6092187, 3753637, 2818150, 2978126, 
                 5166592, 5721297, 3445169, 2820940, 5661619, 
                 2551390, 5892071, 4644447, 6121803, 2792571]

# 겹치는 상품 리뷰 수
overlap_reviews = df[df['goods_no'].isin(overlap_goods)]
only_rec_reviews = df[~df['goods_no'].isin(overlap_goods)]

print(f"전체 리뷰: {len(df)}건")
print(f"\n겹치는 15개 상품 리뷰: {len(overlap_reviews)}건")
print(f"추천순에만 있는 35개 상품 리뷰: {len(only_rec_reviews)}건")

print(f"\n상품별 리뷰 수 (겹치는 15개):")
print(overlap_reviews.groupby('goods_no')['review_no'].count().sort_values(ascending=False))

print(f"\n상품별 리뷰 수 (추천순만 35개):")
print(only_rec_reviews.groupby('goods_no')['review_no'].count().sort_values(ascending=False))

전체 리뷰: 21289건

겹치는 15개 상품 리뷰: 7989건
추천순에만 있는 35개 상품 리뷰: 13300건

상품별 리뷰 수 (겹치는 15개):
goods_no
2792571    1000
2818150    1000
2820940    1000
2978126    1000
3445169    1000
3753637    1000
2551390     997
6092187     389
5166592     174
4644447     159
6092190     146
6121803      40
5721297      31
5661619      28
5892071      25
Name: review_no, dtype: int64

상품별 리뷰 수 (추천순만 35개):
goods_no
1357768    1000
1945314    1000
2124425    1000
2477832    1000
3135346    1000
3863280    1000
2818151     999
1357771     999
3378573     973
3134752     909
3134739     639
3758437     598
3779651     567
3779650     345
4642930     259
4728043     223
4719640     147
5166562     118
5258615      65
5242005      61
6092691      60
4624151      52
5745829      51
5166563      43
5657916      30
5940961      28
5985469      25
5768101      24
5795928      19
5940965      19
5768119      15
6121801      12
5928482       8
5698182       7
6092188       5
Name: review_no, dtype: int64


# 추천순 리뷰 데이터 전처리

## 결측 확인

In [6]:
print(f"전체 결측치 현황:")
print(df.isnull().sum())

전체 결측치 현황:
goods_no             0
review_no            0
content              6
grade                0
option_size         35
size_feedback    17289
create_date          0
review_sex        1657
height               0
weight               0
dtype: int64


In [ ]:
# content 결측 6건 삭제
print(f"처리 전: {len(df)}건")
df = df.dropna(subset=['content'])
print(f"처리 후: {len(df)}건")

처리 전: 21295건
처리 후: 21289건


- `option_size` 35건 : 사이즈 파싱 후 정리
- `size_feedback` 1657건 : 유지. 설문 미응답은 자연스럽기도 하고, 리뷰는 작성해줬기 때문에 유지
- `review_sex` 1657건 : 유지. 판매금액순 리뷰데이터처럼 신뢰도 낮음. (여자코너에서 남자라고 응답)

## 중복 리뷰

In [5]:
print(f"중복 전: {len(df)}건")
print(f"중복 review_no 수: {df['review_no'].duplicated().sum()}건")
print(f"content 완전 일치 중복: {df['content'].duplicated().sum()}건")

중복 전: 21295건
중복 review_no 수: 0건
content 완전 일치 중복: 1038건


## 옵션 사이즈 → 컬러/사이즈로 분리

In [9]:
def parse_option(option):
    if pd.isna(option):
        return None, None
    parts = str(option).split('·')
    if len(parts) == 2:
        color = re.sub(r'^\d+\.', '', parts[0]).strip()
        size = parts[1].strip()
        return color, size
    else:
        size = str(option).strip()
        return None, size

df[['color', 'size']] = df['option_size'].apply(
    lambda x: pd.Series(parse_option(x))
)

print(df[['option_size', 'color', 'size']].head(10))
print(f"\ncolor 결측: {df['color'].isna().sum()}건")
print(f"size 결측: {df['size'].isna().sum()}건")
print(f"\n사이즈 종류: {sorted(df['size'].dropna().unique())}")
print(f"색상 종류: {df['color'].dropna().unique()}")

   option_size color size
0  03.베이지 · 26   베이지   26
1   01.블랙 · 26    블랙   26
2  03.베이지 · 29   베이지   29
3           24  None   24
4   02.크림 · 26    크림   26
5  03.베이지 · 25   베이지   25
6           26  None   26
7  03.베이지 · 27   베이지   27
8           27  None   27
9   01.블랙 · 24    블랙   24

color 결측: 19905건
size 결측: 35건

사이즈 종류: ['22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '34', 'L', 'M', 'S', 'XL', 'XS']
색상 종류: ['베이지' '블랙' '크림' '피치' '페일 그린' '화이트' '미디엄 그레이' '라이트 그레이' '클라우디 블루' '페일 핑크'
 '블랙/실버' '브라운' '블랙/화이트/화이트' '블랙/블랙/블랙' '블랙/미디엄그레이/화이트' '화이트/화이트/화이트'
 '라이트 베이지' '아이보리' '레드 브라운' '핑크' '더스티 민트' '다크 브라운/실버' '브라운/실버' '화이트/실버'
 '블랙/블랙' '블랙/골드']


In [10]:
# Size 결측 삭제
print(f"삭제 전: {len(df)}건")
df = df[df['size'].notna()]
print(f"삭제 후: {len(df)}건")

삭제 전: 21289건
삭제 후: 21254건


## grade 숫자열 확인

In [12]:
print(f"grade 타입: {df['grade'].dtype}")
print(f"grade 고유값: {df['grade'].unique()}")
print(f"\ngrade 분포:\n{df['grade'].value_counts().sort_index()}")

# 별점 없는 리뷰 플래그
df['has_grade'] = df['grade'] != 0
print(f"\n별점 있는 리뷰: {df['has_grade'].sum()}건")
print(f"별점 없는 리뷰(0점): {(~df['has_grade']).sum()}건")

grade 타입: int64
grade 고유값: [5 4 3 1 2 0]

grade 분포:
grade
0      183
1       63
2       88
3      585
4     2697
5    17638
Name: count, dtype: int64

별점 있는 리뷰: 21071건
별점 없는 리뷰(0점): 183건


## 날짜 파싱(parsing)

In [13]:
df['create_date'] = pd.to_datetime(df['create_date'])
df['year'] = df['create_date'].dt.year
df['month'] = df['create_date'].dt.month

print(f"날짜 범위: {df['create_date'].min()} ~ {df['create_date'].max()}")
print(f"\n월별 리뷰 분포:")
print(df.groupby(['year','month'])['review_no'].count())

날짜 범위: 2023-04-25 18:14:34+09:00 ~ 2026-04-21 08:59:08+09:00

월별 리뷰 분포:
year  month
2023  4          10
      5         106
      6          96
      7          52
      8          42
      9          72
      10        145
      11        219
      12        157
2024  1         128
      2         166
      3         253
      4         373
      5         511
      6         783
      7         671
      8         425
      9         451
      10        940
      11        925
      12        609
2025  1         320
      2         319
      3         456
      4         679
      5        1216
      6        1764
      7        1335
      8        1011
      9        1058
      10       1264
      11       1098
      12        533
2026  1         264
      2         479
      3        1054
      4        1270
Name: review_no, dtype: int64


## BMI 컬럼 추가

In [14]:
# BMI 계산
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)
df['bmi'] = df['bmi'].round(1)

print(f"BMI 계산 전 결측: {df['bmi'].isna().sum()}건")
print(f"\nBMI 기술통계:\n{df['bmi'].describe()}")

BMI 계산 전 결측: 1860건

BMI 기술통계:
count    19394.0
mean         inf
std          NaN
min          0.0
25%         18.9
50%         20.2
75%         22.0
max          inf
Name: bmi, dtype: float64


In [15]:
# height, weight 이상치 → NaN
df['height'] = df['height'].where(
    (df['height'] >= 140) & (df['height'] <= 200)
)
df['weight'] = df['weight'].where(
    (df['weight'] >= 30) & (df['weight'] <= 150)
)

# BMI 재계산
df['bmi'] = (df['weight'] / ((df['height'] / 100) ** 2)).round(1)

print(f"\n이상치 처리 후 BMI 기술통계:\n{df['bmi'].describe()}")
print(f"height 결측: {df['height'].isna().sum()}건")
print(f"weight 결측: {df['weight'].isna().sum()}건")
print(f"BMI 결측: {df['bmi'].isna().sum()}건")


이상치 처리 후 BMI 기술통계:
count    19196.000000
mean        20.668514
std          2.943456
min          9.300000
25%         18.900000
50%         20.200000
75%         21.900000
max         66.700000
Name: bmi, dtype: float64
height 결측: 1955건
weight 결측: 1979건
BMI 결측: 2058건


## 리뷰 성별 확인

In [17]:
print(df['review_sex'].value_counts(dropna=False))

review_sex
여성     18687
NaN     1657
남성       910
Name: count, dtype: int64


## 텍스트 클리닝

In [16]:
def clean_text(text):
    if pd.isna(text):
        return None
    # 이모지 및 특수문자 제거
    text = re.sub(r'[^\w\s가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9]', ' ', text)
    # 반복 문자 제거 (3번 이상 → 1번)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    # 영어 소문자 통일
    text = text.lower()
    # 여러 공백 → 단일 공백
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

df['content_clean'] = df['content'].apply(clean_text)

# 5자 이하 리뷰 확인
short = df[df['content_clean'].str.len() <= 5]
print(f"5자 이하 리뷰: {len(short)}건")
print(short['content_clean'].values)

5자 이하 리뷰: 2건
['슬랙스코디' '핏 예뻐요']


In [18]:
print(f"제거 전: {len(df)}건")
df = df[df['content_clean'].str.len() > 5]
print(f"제거 후: {len(df)}건")

제거 전: 21254건
제거 후: 21252건


# 무신사 추천순 top50 상품과 조인

In [19]:
# 추천순 top50에서 필요한 컬럼만 선택
df_products_slim = df_products[['goods_no', 'goods_name', 'sale_price', 'sale_rate', 'review_count', 'review_score']]

# 병합
df = df.merge(df_products_slim, on='goods_no', how='left')

print(f"병합 후 컬럼: {df.columns.tolist()}")
print(f"goods_name 결측: {df['goods_name'].isna().sum()}건")
print(df[['goods_no', 'goods_name', 'content_clean']].head(3))

병합 후 컬럼: ['goods_no', 'review_no', 'content', 'grade', 'option_size', 'size_feedback', 'create_date', 'review_sex', 'height', 'weight', 'color', 'size', 'has_grade', 'year', 'month', 'bmi', 'content_clean', 'goods_name', 'sale_price', 'sale_rate', 'review_count', 'review_score']
goods_name 결측: 0건
   goods_no                           goods_name  \
0   6092187  [한소희 PICK] 우먼즈 코튼 커브드 팬츠 (3 colors)   
1   6092187  [한소희 PICK] 우먼즈 코튼 커브드 팬츠 (3 colors)   
2   6092187  [한소희 PICK] 우먼즈 코튼 커브드 팬츠 (3 colors)   

                                       content_clean  
0  커브드팬츠 베이지 색상까지 구매했습니다 스트레이트 핏이 돌아온다고 하지만 아직 널널...  
1    매우 잘 구겨져서 주의 ㅠㅠ 또 여름에 입기에는 더울 것 같아요 지금도 덥습니다 ㅎㅎ  
2  너무 마음에 듭니다 모양이 화면 그대로 이쁘고 무엇보다 기장이 짧지 않아서 좋고 매...  


## 카테고리 분류

In [20]:
def categorize(name):
    name = str(name)
    if any(k in name for k in ['티셔츠', '셔츠', '탱크 탑']):
        return '상의'
    elif any(k in name for k in ['팬츠', '슬랙스', '데님']):
        return '하의'
    elif any(k in name for k in ['스커트']):
        return '스커트'
    elif any(k in name for k in ['블레이저', '재킷', '윈드브레이커', '코트', '블루종']):
        return '아우터'
    elif any(k in name for k in ['벨트']):
        return '액세서리'
    elif any(k in name for k in ['백', '백팩', '숄더백']):
        return '가방'
    elif any(k in name for k in ['삭스', '브라', '드레스']):
        return '기타'
    else:
        return '기타'

df['category'] = df['goods_name'].apply(categorize)
print(df['category'].value_counts())
print(f"\n기타 항목:")
print(df[df['category'] == '기타']['goods_name'].unique())

category
하의      16155
상의       3107
액세서리     1008
스커트       626
기타        179
아우터       177
Name: count, dtype: int64

기타 항목:
['우먼즈 버티컬 시어삭스 4팩 [블랙]' '우먼즈 버티컬 시어삭스 4팩 [화이트]'
 '[다구매시 할인] 우먼즈 심리스 브라 (7 colors)' '우먼즈 리브드 슬라우치 삭스 4팩 [그레이]'
 '우먼즈 포멀 미니 드레스 [그레이]']


In [21]:
def categorize(name):
    name = str(name)
    if any(k in name for k in ['티셔츠', '셔츠', '탱크 탑']):
        return '상의'
    elif any(k in name for k in ['팬츠', '슬랙스', '데님']):
        return '하의'
    elif any(k in name for k in ['스커트']):
        return '스커트'
    elif any(k in name for k in ['블레이저', '재킷', '윈드브레이커', '코트', '블루종']):
        return '아우터'
    elif any(k in name for k in ['벨트']):
        return '액세서리'
    elif any(k in name for k in ['백', '백팩', '숄더백']):
        return '가방'
    elif any(k in name for k in ['삭스']):
        return '양말'
    elif any(k in name for k in ['브라']):
        return '언더웨어'
    elif any(k in name for k in ['드레스']):
        return '드레스'
    else:
        return '기타'

df['category'] = df['goods_name'].apply(categorize)
print(df['category'].value_counts())
print(f"\n기타 항목:")
print(df[df['category'] == '기타']['goods_name'].unique())

category
하의      16155
상의       3107
액세서리     1008
스커트       626
아우터       177
양말        112
언더웨어       60
드레스         7
Name: count, dtype: int64

기타 항목:
[]


## 사이즈 타입 분류

In [22]:
def size_type(size):
    if pd.isna(size):
        return None
    if str(size).isdigit():
        return '숫자형'
    else:
        return '문자형'

df['size_type'] = df['size'].apply(size_type)
print(df['size_type'].value_counts(dropna=False))
print(f"\n카테고리별 사이즈 타입:")
print(df.groupby(['category', 'size_type'])['review_no'].count())

size_type
숫자형    16139
문자형     5113
Name: count, dtype: int64

카테고리별 사이즈 타입:
category  size_type
드레스       문자형              7
상의        문자형           3107
스커트       문자형            626
아우터       문자형            177
액세서리      숫자형           1008
양말        문자형            112
언더웨어      문자형             60
하의        문자형           1024
          숫자형          15131
Name: review_no, dtype: int64


In [23]:
print(f"최종 데이터: {len(df)}건, {len(df.columns)}개 컬럼")
print(f"컬럼 목록: {df.columns.tolist()}")

df.to_csv("recommend_reviews_merged.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료: recommend_reviews_merged.csv")

최종 데이터: 21252건, 24개 컬럼
컬럼 목록: ['goods_no', 'review_no', 'content', 'grade', 'option_size', 'size_feedback', 'create_date', 'review_sex', 'height', 'weight', 'color', 'size', 'has_grade', 'year', 'month', 'bmi', 'content_clean', 'goods_name', 'sale_price', 'sale_rate', 'review_count', 'review_score', 'category', 'size_type']

저장 완료: recommend_reviews_merged.csv
